In [1]:
import torch
import torch.nn as nn
from gensim.models import Word2Vec
import numpy as np
from sklearn.model_selection import train_test_split
import re
from pathlib import Path

In [2]:
# Dictionary mapping contractions to their full forms
contractions_dict = {
    "he's": "he is",
    "i'm": "I am",
    "you're": "you are",
    "we've": "we have",
    "they've": "they have",
    "don't": "do not",
    "isn't": "is not",
    "it's": "it is",
    "didn't": "did not",
    "aren't": "are not",
    "let's": "let us",
    "couldn't": "could not",
    "wasn't": "was not",
    "weren't": "were not",
    "ain't": "am not",
    "i've": "I have",
    "that's": "that is",
    "i'll": "I will",
    "you'd": "you would",
    "they're": "they are",
    "i won't": "I will not",
    "can't": "cannot",
    "you've": "you have",
    "there's": "there is",
    "won't": "will not",
    "you'll": "you will",
    "doesn't": "does not",
    "must've": "must have",
    "what's": "what is",
    "we're": "we are",
    "haven't": "have not",
    "wouldn't": "would not",
    "i'd": "I would",
    "she's": "she is",
    "nobody's": "nobody is",
    "we'll": "we will",
    "they'd": "they would",
    "mustn't": "must not",
    "could've": "could have",
    "shouldn't": "should not",
    "he'll": "he will",
    "he'd": "he would",
    "hadn't": "had not",
    "where'd": "where did",
    "we'd": "we would",
}

# Function to replace contractions in the text
def replace_contractions(text, contractions_map):
    # Create a regex pattern that matches any of the contractions
    pattern = re.compile(r'\b(' + '|'.join(re.escape(key) for key in contractions_map.keys()) + r')\b')
    # Replace contractions using the dictionary
    return pattern.sub(lambda x: contractions_map[x.group()], text)

# Function to process the text file
def process_text_file(input_file, output_file, contractions_map):
    # Read the contents of the input file
    with open(input_file, 'r') as file:
        text = file.read().lower()

    # Replace contractions
    new_text = replace_contractions(text, contractions_map)

    # Write the modified text to the output file
    with open(output_file, 'w') as file:
        file.write(new_text)

    return new_text

# Specify the input and output file paths
input_file = 'adele.txt'  # Replace with the actual file path
output_file = 'output.txt'

# Process the text file if not present
if not Path(output_file).exists():
    modified_texte = process_text_file(input_file, output_file, contractions_dict)
    print("Contractions replaced and saved to", output_file)
else: 
    with open(output_file, "r") as f:
        modified_texte = f.read().lower()


# Parameters

In [3]:
# It is arbitrary values
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
N_LAYERS = 2
DROPOUT = 0.5
N_EPOCHS = 150
LR = 3e-4
BATCH_SIZE = 32
SEQ_LEN = 30

# Tokenization

In [4]:
file_path = "output.txt"

with open(file_path, 'r') as file:
    text = file.readlines()

text = modified_texte.split("\n")

sentences = []
for line in text:
    # print(line)

    # Remove all non-alphanumeric characters and convert to lowercase
    clean_line = re.sub(r'[^\w\s]', '', line.lower())
    
    # TODO: faire en sorte que ca soit propre, y a des phrase d'un seul mot et c'est vraiment nul
    #       J me suis pas concentré sur ca pour le moment. J'ai aussi l'impression que la ponctuation reste dans le bail donc chelou
    
    words = clean_line.split() 

    if len(words) > 1:
        # Add the EOS token at the end of sentence
        words.insert(0,"<SOS>")
        words.append("<EOS>")
        sentences.append(words)
    else:
        print("Phrase ignored:", words)

# Vector size of 100, it can be modified, we can play with the parameters of Word2Vec
word2vec_model = Word2Vec(sentences, vector_size=EMBEDDING_DIM, window=30, min_count=1, epochs=100)
word2vec_model.save("word2vec100_adele.model")

word_vector = word2vec_model.wv["the"]
print(word_vector)

Phrase ignored: ['sweetest']
Phrase ignored: ['sweetest']
Phrase ignored: ['sweetest']
Phrase ignored: ['darling']
Phrase ignored: ['ooh']
Phrase ignored: ['depleted']
Phrase ignored: ['hustle']
[-1.3460292   0.370656    0.1056833  -0.18009259  0.23197673  1.4486597
  0.13887027 -0.35513687 -0.9351932   1.1105744   0.4971323   1.1907109
  0.77684    -0.24007979 -1.0641501  -0.6375441  -1.3049464   0.17120427
  0.6549169   0.4197069   0.8191462  -0.60478866  0.21033826  2.0023656
  0.12651698 -1.3388166   0.12129583  0.3234919  -0.5908935   0.5603385
 -0.9756329   1.5728955   1.0196143   0.0216487   0.05205343 -0.3873211
 -0.6233349   0.03420864  0.3177574   0.22946239  0.30964366 -0.64956766
 -0.02507755  0.3726206   0.22518322 -1.4765748  -0.4293024   0.2932471
  0.2692607  -0.34330875  0.31591168 -0.14364891  0.8001206  -0.8527683
 -1.1313667  -0.3512706  -1.944987    0.03847616 -0.33975974 -0.2076009
  0.91199327 -0.3080901   0.97450995  0.07963515 -1.9533216   0.6398713
  0.7429158

In [5]:
print(word2vec_model.wv.most_similar("baby"))

[('miss', 0.5177789926528931), ('details', 0.4563436210155487), ('bore', 0.4527880549430847), ('down', 0.4368474781513214), ('give', 0.4363813102245331), ('lights', 0.43516236543655396), ('off', 0.42451873421669006), ('setting', 0.42244189977645874), ('tone', 0.42213642597198486), ('do', 0.42170003056526184)]


In [6]:
def sentences_to_vectors(sentences, word2vec_model):
    """Convert the sentences to the vector learned by word2vec

    Args:
        sentences (List[List[str]]): The first list contain the lines/sentences, the second one contain the words of the sentences
        word2vec_model (Word2Vec): Word2Vec object trained on the actual corpus

    Returns:
        List[List[torch.Tensor]]: The same 2 list with the words converted to their word2Vec vector
    """
    indices = []
    for sentence in sentences:
        sentence_vectors = []
        for word in sentence: # Si le mot est dans le vocabulaire
            if word in word2vec_model.wv.key_to_index:  
                sentence_vectors.append(word2vec_model.wv[word])
            else: # Si le mot n'existe pas dans le vocabulaire
                print(f"Le mot '{word}' n'existe pas dans le vocabulaire.")
                exit(0) 
        indices.append(torch.tensor(np.array(sentence_vectors)))
    return indices

# Conversion des phrases en vecteur
sentence_vectors = sentences_to_vectors(sentences, word2vec_model)

print(f"Nb of sentences: {len(sentence_vectors)}")
print(f"Nb of words in first sentence: {len(sentence_vectors[0])}")
print(f"Embbedding size of first word: {len(sentence_vectors[0][0])}")

# Séparer les données en ensembles d'entraînement et de validation
train_vectors, val_vectors = train_test_split(sentence_vectors, test_size=0.2, random_state=42)
train_sentences, val_sentences = train_test_split(sentences, test_size=0.2, random_state=42)


def check_mapping(train_vectors, val_vectors, train_sentences, val_sentences):
    """ Check via the size that all vectors are correctly mapped to the right sentence

    Args:
        train_vectors (List[List[torch.Tensor]]): list of vector of check
        val_vectors (List[List[torch.Tensor]]): list of vector of check
        train_sentences (List[List[str]): list of words to check
        val_sentences (List[List[str]): list of words to check

    Raises:
        Exception: if the check fails
    """
    test_vectors = [train_vectors, val_vectors]
    test_sentences = [train_sentences, val_sentences]

    for i in range(len(test_vectors)):
        for index in range(len(test_vectors[i])):
            if (len(test_vectors[i][index]) != len(test_sentences[i][index])):
                raise Exception("The size of the vector isn't the same that the corresponding sentence")

check_mapping(train_vectors, val_vectors, train_sentences, val_sentences)

Nb of sentences: 2393
Nb of words in first sentence: 6
Embbedding size of first word: 100


In [7]:
vocab_size = len(word2vec_model.wv.key_to_index)
print(vocab_size)

1346


# Model

In [11]:
class LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers):
        super(LSTM, self).__init__()

        self.lstm = nn.LSTM(embedding_dim, vocab_size + hidden_dim, n_layers, proj_size= vocab_size)

    def forward(self, x):
        x, _  = self.lstm(x)
        # print(x.shape)
        return x
    
lstm = LSTM(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS)

In [8]:
class NLP(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers, dropout, model_type='LSTM'):
        super(NLP, self).__init__()
        self.vocab_size = vocab_size
        self.num_layers = n_layers
        self.rnn_type = model_type
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.hidden_dim = hidden_dim

        if model_type == 'LSTM':
            self.nlp = nn.LSTM(embedding_dim, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        elif model_type == 'GRU':
            self.nlp = nn.GRU(embedding_dim, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        else:
            raise Exception("Model type not supported")
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden):
        # x = [batch_size, seq_len, vocab_size]
        # print(x.shape)
        x, hidden = self.nlp(x)
        # x = [batch_size, seq_len, state_dim]
        # print(x.shape)
        x = self.fc(x)
        # x = [batch_size, seq_len, vocab_size]
        # print(x.shape)
        return x, hidden
    
    def init_hidden(self, batch_size):
        if self.rnn_type == 'LSTM':
            # LSTM requires both hidden state and cell state
            hidden = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(self.device)
            cell = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(self.device)
            return (hidden, cell)
        else:
            # GRU only requires the hidden state
            hidden = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(self.device)
            return hidden

In [9]:
def train(model, vectors, sentences, n_epochs, lr, batch_size, seq_len, name):
    # Setup GPU related variables
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device = {device}")
    torch.cuda.empty_cache()
    model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(reduction='mean')

    model.train()

    for epoch in range(n_epochs):
        train_losses = []
        for i,sentence in enumerate(sentences):
            hidden = model.init_hidden(batch_size)

            # Substract the last word as we didn't include the eos token (so we couldn't predict it)
            x = vectors[i][:-1]
            
            # Create the one hot encoding of the correct prediction
            y = torch.tensor(np.zeros((len(x), vocab_size)))
            
            # Start the sequence to the first word as we didn't include the sos token (so we couldn't predict it)
            for j in range(1, len(x)):
                y[0][word2vec_model.wv.get_index(sentence[j])] = 1.0

            optimizer.zero_grad()
            #print(sentence)
            #print(x.shape)
            output, hidden = model(x, hidden)

            loss = criterion(output, y)
            train_losses.append(loss.cpu().detach())
            loss.backward()
            optimizer.step()

            if i % 100 == 0:
                print(f"Epoch {epoch}, step {i}, loss {loss.item()}")
        
        print(f"Epoch {epoch} finished. Train loss: {np.array(train_losses).mean()}, Perplexity: {np.exp(np.array(train_losses).mean())}")
        
    torch.save(model.state_dict(), f"model_save/{name}.pth")

In [14]:
#model_type = 'GRU'
#GRU_model = NLP(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, model_type)

#train(GRU_model, train_vectors, train_sentences, N_EPOCHS, LR, BATCH_SIZE, SEQ_LEN, f"{model_type}_model_Word2Vec{EMBEDDING_DIM}")

In [10]:
model_type = "LSTM"
LSTM_model = NLP(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, model_type)

train(LSTM_model, train_vectors, train_sentences, N_EPOCHS, LR, BATCH_SIZE, SEQ_LEN, f"{model_type}_model_Word2Vec{EMBEDDING_DIM}")

device = cpu
Epoch 0, step 0, loss 6.407360765669081
Epoch 0, step 100, loss 5.649939673287528
Epoch 0, step 200, loss 4.588614290410822
Epoch 0, step 300, loss 4.841103255748749
Epoch 0, step 400, loss 4.425870206620958
Epoch 0, step 500, loss 5.5513598918914795
Epoch 0, step 600, loss 5.2600353240966795
Epoch 0, step 700, loss 5.437628613577949
Epoch 0, step 800, loss 4.646421167585585
Epoch 0, step 900, loss 5.045084688398573
Epoch 0, step 1000, loss 3.890139857927958
Epoch 0, step 1100, loss 3.5912411411603293
Epoch 0, step 1200, loss 4.617036056518555
Epoch 0, step 1300, loss 4.827496439218521
Epoch 0, step 1400, loss 4.792142105102539
Epoch 0, step 1500, loss 4.510025048255921
Epoch 0, step 1600, loss 4.740588039159775
Epoch 0, step 1700, loss 3.918698032697042
Epoch 0, step 1800, loss 5.406726102034251
Epoch 0, step 1900, loss 4.644688606262207
Epoch 0 finished. Train loss: 4.748792559266503, Perplexity: 115.44480757591906
Epoch 1, step 0, loss 4.801108148362902
Epoch 1, step 10

In [27]:
def generate_text(model, encoder: Word2Vec, start_word, max_words=20, random_sample=False):
    """
    Generate text based on the trained model output.
    
    Parameters:
    - model: The trained PyTorch model.
    - encoder: The OneHotEncoder used for encoding the words.
    - start_word: The initial word to start generating text.
    - max_words: Number of words to generate.
    - random_sample: If True, sample from the distribution instead of taking the max probability.
    
    Returns:
    - generated_text: The generated sequence of words.
    """
    model.eval()

    start_word = start_word.lower().split()
    # Initialize the generated text with the start word
    generated_words = start_word
    
    # Convert the start word to its one-hot encoded representation
    
    input_tensor = [encoder.wv.get_vector("<SOS>")]
    
    for word in start_word:
        input_tensor.append(encoder.wv.get_vector(word))
    input_tensor = torch.Tensor(input_tensor).unsqueeze(0).to(model.device)  # Add batch dimension

    # Initialize hidden state
    hidden = model.init_hidden(batch_size=1)
    
    # Generate the specified number of words
    while(True):
        # Get the model output with the hidden state
        with torch.no_grad():
            output, hidden = model(input_tensor, hidden)  # Pass hidden state

        # Apply softmax to get probabilities
        probabilities = torch.softmax(output, dim=-1).squeeze().cpu().numpy()
        
        # Resize probabilities in case of single word given as input
        if len(probabilities.shape) != 2:
            probabilities = probabilities.reshape((1,-1))

        for probability in probabilities:
            # Determine the next word
            if random_sample:
                next_index = np.random.choice(len(probability), p=probability)
            else:
                next_index = np.argmax(probability)

            # Get the corresponding word from the encoder
            next_word = encoder.wv.index_to_key[next_index]

            stop = False
            if next_word == "<EOS>" or len(generated_words) > max_words:
                stop = True
                break
            else:   
                generated_words.append(next_word)

        if stop:
            break
        
        # Update the input tensor with the new word
        input_tensor = [encoder.wv.get_vector("<SOS>")]
        for word in generated_words:
            input_tensor.append(encoder.wv.get_vector(word))
        
        input_tensor = torch.Tensor(input_tensor).unsqueeze(0).to(model.device)  # Add batch dimension
    
    # Join the generated words into a single string
    generated_text = ' '.join(generated_words)
    return generated_text

In [28]:
# Exemple d'utilisation
generated_text = generate_text(LSTM_model, word2vec_model, start_word='who', random_sample=False)
print(generated_text)

generated_text = generate_text(LSTM_model, word2vec_model, start_word='had some', random_sample=False)
print(generated_text)

generated_text = generate_text(LSTM_model, word2vec_model, start_word='every time he am', random_sample=False)
print(generated_text)

generated_text_random = generate_text(LSTM_model, word2vec_model, start_word='Baby', random_sample=True)
print(generated_text_random)

who you you you you you you you you you you you you you you you you you you you you
had some you you you you you you you you you you you you you you you you you you you
every time he am you you you you you you you you you you you you you you you you you
baby a your love on am you shows for you to whenever goes i to pushed would she me one you
